# Marketing A/B Test: Did the Ad Campaign Work?
**Business question:** Users were randomly shown either the company's **ad** or a neutral **public service announcement (PSA)**. Did the ads cause more purchases? If so, by how much, and when do they work best?

**Data:** Kaggle [Marketing A/B Testing](https://www.kaggle.com/datasets/faviovaz/marketing-ab-testing), file `marketing_AB.csv` (about 588K users).

**Methods:** two-proportion z-test, confidence intervals, minimum detectable effect, dose-response and timing analysis.

## 1. Upload the CSV
Run the cell below and choose **marketing_AB.csv**.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
df = pd.read_csv("marketing_AB.csv")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df = df.drop(columns=[c for c in df.columns if c.startswith("unnamed")])
if df["converted"].dtype == object:
    df["converted"] = df["converted"].astype(str).str.lower().eq("true")
df["converted"] = df["converted"].astype(int)
print(df.shape)
print("Duplicate users:", df["user_id"].duplicated().sum())
df.head()

## 2. Group sizes and conversion rates

In [ ]:
summary = (df.groupby("test_group")
             .agg(users=("user_id", "count"), conversions=("converted", "sum"))
             .assign(conversion_rate=lambda t: t["conversions"] / t["users"]))
summary["share_of_users"] = summary["users"] / summary["users"].sum()
summary.style.format({"users": "{:,}", "conversions": "{:,}", "conversion_rate": "{:.3%}", "share_of_users": "{:.1%}"})

The split is intentionally uneven (most users see ads; a small control group sees the PSA).
That's fine for the test as long as assignment was random. We just need the right statistics for unequal groups.

## 3. Is the difference statistically significant?

In [ ]:
n_ad, x_ad = summary.loc["ad", ["users", "conversions"]]
n_psa, x_psa = summary.loc["psa", ["users", "conversions"]]
p_ad, p_psa = x_ad / n_ad, x_psa / n_psa

# Two-proportion z-test (pooled)
p_pool = (x_ad + x_psa) / (n_ad + n_psa)
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1 / n_ad + 1 / n_psa))
z = (p_ad - p_psa) / se_pool
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

# 95% confidence interval for the difference (unpooled)
se_diff = np.sqrt(p_ad * (1 - p_ad) / n_ad + p_psa * (1 - p_psa) / n_psa)
diff = p_ad - p_psa
ci_low, ci_high = diff - 1.96 * se_diff, diff + 1.96 * se_diff
lift = diff / p_psa

print(f"Ad conversion:   {p_ad:.3%}")
print(f"PSA conversion:  {p_psa:.3%}")
print(f"Difference:      {diff*100:+.3f} percentage points (95% CI {ci_low*100:+.3f} to {ci_high*100:+.3f})")
print(f"Relative lift:   {lift:+.1%}")
print(f"z = {z:.2f}, p-value = {p_value:.2e}")
print("Significant at 5%" if p_value < 0.05 else "NOT significant at 5%")

In [ ]:
# Minimum detectable effect: the smallest lift this test could reliably detect (80% power, 5% significance)
z_a, z_b = stats.norm.ppf(0.975), stats.norm.ppf(0.80)
mde = (z_a + z_b) * np.sqrt(p_psa * (1 - p_psa) * (1 / n_ad + 1 / n_psa))
print(f"Minimum detectable difference: {mde*100:.3f} pp ({mde/p_psa:.1%} relative)")
print(f"Observed difference:           {diff*100:.3f} pp, so the test was {'well' if diff > mde else 'NOT'} powered to detect it.")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
rates = [p_psa, p_ad]
errs = [1.96 * np.sqrt(p * (1 - p) / n) for p, n in [(p_psa, n_psa), (p_ad, n_ad)]]
ax.bar(["PSA (control)", "Ad"], rates, yerr=errs, capsize=8, color=["#9db4d6", "#1f6feb"])
for i, r in enumerate(rates):
    ax.text(i, r * 1.04, f"{r:.2%}", ha="center", fontweight="bold")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
ax.set_title(f"Conversion rate with 95% CI (lift {lift:+.0%})")
plt.tight_layout()
plt.savefig("ab_conversion.png", dpi=150)
plt.show()

## 4. Business impact: incremental conversions caused by the ads

In [ ]:
incremental = diff * n_ad
print(f"Ads caused about {incremental:,.0f} extra conversions among {n_ad:,} users who saw them")
print(f"(95% CI: {ci_low*n_ad:,.0f} to {ci_high*n_ad:,.0f}).")

AOV = 50   # assumed average order value in $; change this if you know the real number
print(f"At an assumed ${AOV} order value, that is about ${incremental*AOV:,.0f} in incremental revenue.")

## 5. Does seeing more ads help? (dose-response)

In [ ]:
bins = [0, 10, 25, 50, 100, 200, np.inf]
labels = ["1–10", "11–25", "26–50", "51–100", "101–200", "200+"]
df["ads_bucket"] = pd.cut(df["total_ads"], bins=bins, labels=labels)
dose = df.pivot_table(index="ads_bucket", columns="test_group", values="converted", aggfunc="mean", observed=True)
dose["lift_pp"] = (dose["ad"] - dose["psa"]) * 100
dose["users"] = df.groupby("ads_bucket", observed=True).size()

fig, ax = plt.subplots(figsize=(8, 4))
dose[["psa", "ad"]].plot(ax=ax, marker="o", color=["#9db4d6", "#1f6feb"])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("Number of ads/PSAs seen")
ax.set_ylabel("Conversion rate")
ax.set_title("Conversion rises with exposure, and faster for ads")
plt.tight_layout()
plt.savefig("ab_dose_response.png", dpi=150)
plt.show()
dose.round(4)

Note: exposure isn't randomised (more engaged users see more ads), so this pattern is **correlation**, not proof that more ads cause more sales. The ad-vs-PSA gap *within* each bucket is the fairer comparison.

## 6. When do ads work best? (day and hour)

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = df.pivot_table(index="most_ads_day", columns="test_group", values="converted", aggfunc="mean").reindex(day_order)
by_day["lift_pp"] = (by_day["ad"] - by_day["psa"]) * 100

by_hour = df[df["test_group"] == "ad"].groupby("most_ads_hour")["converted"].agg(["mean", "count"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
by_day["lift_pp"].plot.bar(ax=axes[0], color="#1f6feb")
axes[0].set_title("Ad lift over PSA by day (percentage points)")
axes[0].set_xlabel("")
by_hour["mean"].plot(ax=axes[1], marker="o", color="#14213d")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
axes[1].set_title("Ad-group conversion by hour of peak exposure")
axes[1].set_xlabel("Hour of day")
plt.tight_layout()
plt.savefig("ab_timing.png", dpi=150)
plt.show()
by_day.round(4)

## 7. Key numbers for your README

In [ ]:
best_day = by_day["lift_pp"].idxmax()
worst_day = by_day["lift_pp"].idxmin()
busy_hours = by_hour[by_hour["count"] >= 1000]
best_hour = busy_hours["mean"].idxmax()

print(f"- {len(df):,} users; {summary.loc['ad','share_of_users']:.0%} saw ads, {summary.loc['psa','share_of_users']:.0%} saw a PSA (control).")
print(f"- Ads converted at {p_ad:.2%} vs {p_psa:.2%} for PSA: a {lift:+.0%} relative lift (p = {p_value:.1e}).")
print(f"- 95% CI for the lift: {ci_low*100:+.2f} to {ci_high*100:+.2f} percentage points.")
print(f"- About {incremental:,.0f} incremental conversions attributable to the ads.")
print(f"- Biggest lift on {best_day} ({by_day.loc[best_day,'lift_pp']:+.2f} pp); smallest on {worst_day} ({by_day.loc[worst_day,'lift_pp']:+.2f} pp).")
print(f"- Best hour for ad-group conversion (among hours with 1,000+ users): {best_hour}:00 ({busy_hours.loc[best_hour,'mean']:.2%}).")

In [ ]:
files.download("ab_conversion.png")
files.download("ab_dose_response.png")
files.download("ab_timing.png")